In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import asyncio
import os
import pandas as pd
import copy

from hfppl.llms import CachedCausalLM
from hfppl.inference import smc_standard
from hfppl.distributions import LMContext

from battleship.board import Board, TRIAL_IDS
from battleship.prompting import QuestionGenerationPrompt, TranslationPrompt
from battleship.scoring import compute_score
from battleship.models import QuestionGenerationModel, SingleStepQuestionGenerationModel

In [3]:
# Initialize the HuggingFace model
lm = CachedCausalLM.from_pretrained("codellama/CodeLlama-7b-hf")
# lm = CachedCausalLM.from_pretrained("codellama/CodeLlama-13b-hf")

/home/ubuntu/battleship/.venv/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:671: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/ubuntu/battleship/.venv/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/ubuntu/battleship/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [4]:
async def run_one_step_baseline(
    n_particles=5,
    batch_size=1,
    trial_ids=TRIAL_IDS,
    board_format="textual",
    include_board=False,
    include_instructions=False,
    include_system_prompt=False,
    # question prompt
    q_n_example_trials=3,
    q_n_examples_per_trial=3,
    q_cache_prompt=False,
    # translation prompt
    t_n_example_trials=10,
    t_n_examples_per_trial=1,
    t_cache_prompt=False,
    results_file="hfppl_results.csv",
    random_seed=123,
    verbose=False,
):
    results_all = []
    for trial_id in trial_ids:
        print("-" * 80)
        print(f"Trial {trial_id}")
        print("-" * 80)


        lm.batch_size = batch_size

        assert n_particles % batch_size == 0
        n_batches = n_particles // batch_size

        final_particles = []

        for i in range(n_batches):
            lm.clear_cache()
            lm.clear_kv_cache()

            question_prompt = QuestionGenerationPrompt(
                target_trial_id=trial_id,
                board_format=board_format,
                n_example_trials=q_n_example_trials,
                n_examples_per_trial=q_n_examples_per_trial,
                include_board=include_board,
                include_instructions=include_instructions,
                include_system_prompt=include_system_prompt,
                random_seed=None, # TODO: fix this
            )

            translation_prompt = TranslationPrompt(
                target_trial_id=trial_id,
                n_example_trials=t_n_example_trials,
                n_examples_per_trial=t_n_examples_per_trial,
                include_board=include_board,
                include_instructions=include_instructions,
                include_system_prompt=include_system_prompt,
                random_seed=None,
            )

            print("-" * 80)
            print("QUESTION PROMPT")
            print("-" * 80)
            print(str(question_prompt))
            print("-" * 80)
            print("TRANSLATION PROMPT")
            print("-" * 80)
            print(str(translation_prompt))
            print("-" * 80)

            # Caching speeds up performance, but may result in CUDA out of memory error.
            if q_cache_prompt:
                lm.cache_kv(lm.tokenizer.encode(str(question_prompt)))
            if t_cache_prompt:
                # Additionally, this currently degrades the quality of the translations for an unknown reason.
                lm.cache_kv(lm.tokenizer.encode(str(translation_prompt)))

            model = SingleStepQuestionGenerationModel(
                lm=lm,
                board=Board.from_trial_id(trial_id),
                question_prompt=str(question_prompt),
                translation_prompt=str(translation_prompt),
                verbose=verbose,
            )

            particles = [copy.deepcopy(model) for _ in range(batch_size)]

            while any(map(lambda p: not p.done_stepping(), particles)):
                await asyncio.gather(
                    *[p.step() for p in particles if not p.done_stepping()]
                )

            final_particles += particles

        particles = final_particles

        results_trial = []
        for i, p in enumerate(particles):
            df_p = pd.DataFrame(p.get_final_results())
            df_p["particle"] = i
            df_p["weight"] = p.weight
            results_trial.append(df_p)
        df_trial = pd.concat(results_trial).reset_index(drop=True)
        df_trial["trial_id"] = trial_id
        results_all.append(df_trial)
        df_results = pd.concat(results_all).reset_index(drop=True)
        df_results.to_csv(results_file, index=False)
    return df_results

In [5]:
# RUN SMC BASELINE
df_results = await run_one_step_baseline(
    n_particles=100,
    batch_size=5,
    trial_ids=range(1, 19),
    board_format="textual",
    include_board=True,
    include_instructions=True,
    include_system_prompt=False,
    q_n_example_trials=3,
    q_n_examples_per_trial=10,
    t_n_example_trials=12,
    t_n_examples_per_trial=1,
    q_cache_prompt=True,
    t_cache_prompt=False,
    results_file="results/eval_one_step.csv",
    verbose=True,
)

--------------------------------------------------------------------------------
Trial 3
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
QUESTION PROMPT
--------------------------------------------------------------------------------
You are playing the board game Battleship. There are three ships on the board: Red, Blue, and Purple. Ships are oriented either horizontally or vertically and can be 2, 3, or 4 tiles in length. The board is a 6x6 grid, with numbered rows 1, 2, 3, 4, 5, 6 and lettered columns A, B, C, D, E, F. Coordinates are specified as a row, column pair. For example, 2-C is the tile in row 2, column C.

You will be given a partially-revealed game board. Your task is to ask a single question that will help you gain information about the position of the remaining hidden ships on the board. You can ask any question, but it must be answerable with a single word 

OutOfMemoryError: CUDA out of memory. Tried to allocate 300.00 MiB. GPU 0 has a total capacty of 22.19 GiB of which 161.50 MiB is free. Including non-PyTorch memory, this process has 22.03 GiB memory in use. Of the allocated memory 21.04 GiB is allocated by PyTorch, and 688.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
df_results.sort_values("score", ascending=False)

,prefix,completion,translation,score,type,particle,weight,trial_id
9,What color is at 3A?,What color is at 3A?,(color 3A),1.114823,final,9,1.114823,13
0,What color is at 5A?,What color is at 5A?,(color 5A),1.097786,final,0,1.097786,13
3,Is the purple ship horizontal?,Is the purple ship horizontal?,(== (orient Purple) H),0.939736,final,3,0.939736,13
1,Is the blue ship 2 tiles long?,Is the blue ship 2 tiles long?,(== (size Blue) 2),0.907894,final,1,0.907894,13
7,Is there a ship at 5C?,Is there a ship at 5C?,(not (== (color 5C) Water)),0.514237,final,7,0.514237,13
8,What color is at 5C?,What color is at 5C?,(color 5C),0.514237,final,8,0.514237,13
2,Is the red ship 3 tiles long?,Is the red ship 3 tiles long?,(== (size Red) 3),0.000000,final,2,0.000000,13
4,Is there any ship at 3A?,Is there any ship at 3A?,(any (map (lambda x0 (any (map (lambda y0 (== ...,0.000000,final,4,0.000000,13
5,What color is at 2D?,What color is at 2D?,(color 2D),0.000000,final,5,0.000000,13
6,Is the red ship horizontal?,Is the red ship horizontal?,(== (orient Red) H),0.000000,final,6,0.000000,13
